In [98]:
import pandas as pd
import numpy as np
import re, os
from tqdm.notebook import tqdm
from collections import Counter

In [205]:
na_tokens = ['-', '–', '—', '−', '', 'NA', 'N/A', 'na', 'null', 'NULL']
df_ag_raw = pd.read_excel('NCI60_U95(A_E).xls', na_values=na_tokens, keep_default_na=True)
df_ngs_raw = pd.read_excel('NCI60_NGS.xls', na_values=na_tokens, keep_default_na=True)

In [206]:
gene_ag = Counter(df_ag_raw['Entrez gene id e'].values)
gene_ngs = Counter(df_ngs_raw['Entrez gene id e'].values)

In [207]:
unique_entrez = set([k for k, v in gene_ag.items() if v ==1]) & set([k for k, v in gene_ngs.items() if v ==1])
len(unique_entrez)

6521

In [208]:
df_ag = df_ag_raw[df_ag_raw['Entrez gene id e'].isin(unique_entrez)].sort_values(['Entrez gene id e',])
df_rs= df_ngs_raw[df_ngs_raw['Entrez gene id e'].isin(unique_entrez)].sort_values(['Entrez gene id e',])

In [209]:
df_rs = df_rs.transpose()
df_ag = df_ag.transpose()

In [210]:
gene_name = df_ag.loc['Gene name d',:]
df_ag.columns = gene_name
df_rs.columns = gene_name
len(set(gene_name))

6521

In [211]:
df_ag = df_ag.iloc[7:,:]
df_rs = df_rs.iloc[6:,:]
print(df_ag.shape, df_rs.shape)

(60, 6521) (60, 6521)


In [212]:
ok_rows_ag = ~df_ag.isna().any(axis=1)
ok_rows_rs = ~df_rs.isna().any(axis=1)
common_idx = df_ag.index[ok_rows_ag].intersection(df_rs.index[ok_rows_rs])
common_idx = common_idx.sort_values()
df_ag_clean = df_ag.loc[common_idx].copy()
df_rs_clean = df_rs.loc[common_idx].copy()

In [213]:
df_ag_clean.index = [re.sub(r'[:\-\/\\]','_',x) for x in df_ag_clean.index]
df_rs_clean.index = [re.sub(r'[:\-\/\\]','_',x) for x in df_rs_clean.index]

In [214]:
df_ag_clean.tail()

Gene name d,NAT1,NAT2,SERPINA3,AADAC,AAMP,AANAT,AARS,ABCA3,ABCB7,ABCF1,...,TTN-AS1,TCEB3-AS1,MICA,LINC00837,SEPT5-GP1BB,PINK1-AS,LINC00563,NALCN-AS1,LOC101929524,LOC103344931
RE_CAKI_1,3.675,2.711,5.438,2.525,6.825,7.233,8.206,5.045,4.291,6.441,...,2.537,5.555,7.545,4.071,5.328,5.906,4.336,4.723,4.603,3.686
RE_RXF 393,3.264,2.569,5.456,2.966,6.391,7.086,7.587,5.802,4.025,6.231,...,3.162,6.026,6.966,3.85,5.458,6.384,4.338,4.462,4.593,4.016
RE_SN12C,3.625,2.677,5.646,2.558,7.565,7.105,8.715,4.77,3.965,6.364,...,2.547,5.658,8.04,3.99,5.499,6.33,4.229,4.649,4.598,3.935
RE_TK_10,3.626,2.683,5.941,2.603,7.476,7.07,8.014,6.013,3.902,6.102,...,2.574,5.553,7.102,3.788,5.902,6.337,4.289,4.783,4.776,3.785
RE_UO_31,4.065,2.566,5.608,2.485,6.855,7.006,8.003,6.552,3.459,5.869,...,2.631,5.718,7.759,4.082,5.211,6.12,4.458,4.678,4.557,4.145


In [215]:
print(df_ag_clean.shape, df_rs_clean.shape)

(60, 6521) (60, 6521)


In [216]:
df_ag_clean.to_csv('NCI603_ag.tsv', sep="\t")
df_rs_clean.to_csv('NCI603_rs.tsv', sep="\t")